# Machine Learning Model Training

This notebook trains a machine learning model to predict if a molecule is biologically active based on the extracted features.

**Goal:** Build and evaluate a classifier that can predict molecular activity (ACTIVE = 1 or 0) using the chemical features extracted in the previous notebook.


### Import Required Libraries

- **pandas/numpy**: Data manipulation
- **seaborn/matplotlib**: Data visualization
- **sklearn**: Machine learning tools (models, cross-validation, metrics)
- **ydata_profiling**: Data profiling and analysis


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from ydata_profiling import ProfileReport

#ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load Feature Data

Load the training features extracted in the previous notebook (`training_features.csv`).

This dataset contains ~163 feature columns plus the target variable (ACTIVE).


In [3]:
all_data = pd.read_csv("training_features.csv")

## Step 2: Data Exploration

Check for missing values and examine the dataset structure.

- **Null Values**: Identify any missing data that needs to be handled
- **Shape**: See how many samples and features we have
- **Info**: Check data types and memory usage


In [4]:
pd.DataFrame({'Null Values':all_data.isnull().sum()})

AttributeError: 'Index' object has no attribute '_format_flat'

                                     Null Values
Unnamed: 0                                     0
INDEX                                          0
SMILES                                         0
ACTIVE                                         0
MolFromSmiles                                  0
...                                          ...
CalcNumSpiroAtoms                              0
CalcNumUnspecifiedAtomStereoCenters            0
CalcPhi                                        0
CalcTPSA                                       0
_CalcMolWt                                     0

[164 rows x 1 columns]

In [5]:
all_data.shape

(202895, 164)

In [6]:
all_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202895 entries, 0 to 202894
Columns: 164 entries, Unnamed: 0 to _CalcMolWt
dtypes: float64(25), int64(137), object(2)
memory usage: 253.9+ MB


Set random seed for reproducibility. This ensures that results are consistent across runs.


In [7]:
seed = 20231124
np.random.seed(seed)

## Step 3: Train Machine Learning Model

Train a **Random Forest Classifier** to predict molecular activity.

**Process:**
1. **Separate features (X) from target (y)**: 
   - X: All feature columns (excluding ACTIVE, SMILES, INDEX, MolFromSmiles)
   - y: ACTIVE column (0 = inactive, 1 = active)

2. **10-Fold Cross-Validation**: 
   - Split data into 10 folds
   - Train on 9 folds, test on 1 fold
   - Repeat 10 times with different splits
   - This gives a robust estimate of model performance

3. **Random Forest Classifier**:
   - Ensemble method that combines multiple decision trees
   - Good baseline model for structured data
   - Handles many features well

**Output:** Mean accuracy across all 10 folds (~95.3% in this case)


In [8]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
accuracies = []

# split to training and validation sets
y = all_data["ACTIVE"]
X = all_data.drop(columns=["ACTIVE", "SMILES", "INDEX", "MolFromSmiles"])
from sklearn.model_selection import KFold
kf = KFold(n_splits=10)

count = 0

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Choose a model (RandomForest is a good baseline)
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    accuracies.append(acc)
    print(f"Done with {(count / 10) * 100} %")

print("Accuracy per fold:", accuracies)
print("Mean CV accuracy:", sum(accuracies) / len(accuracies))


Done with 0.0 %
Done with 0.0 %
Done with 0.0 %
Done with 0.0 %
Done with 0.0 %
Done with 0.0 %
Done with 0.0 %
Done with 0.0 %
Done with 0.0 %
Done with 0.0 %
Accuracy per fold: [0.9524396254312469, 0.9517989157220306, 0.9555446032528339, 0.9532281912272055, 0.9518482010842779, 0.951254374291488, 0.9516979644142146, 0.9523879934940115, 0.9546059441076445, 0.9521908423283553]
Mean CV accuracy: 0.9526996655353308
